# 🫀 퀘스트 46 · Q7-T — **자를 정답에 맞춘다: P delineation + QRST 소거**

| | **MedKOS / `notebooks/quest46_q7t_p_anchored.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0067`(Q7-R) · `ailab-2026-0066`(Q7-Q) |
| 규약 | **R16 · R17 · R22 · R27 ③ · R29 ① ② · R30 ① · R33 ① · R34 ① ④ ⑤** |
| 학습 | **0회** — 신호처리만 · GPU 불필요 |

## 이건 주제 변경이 아니라 **도구 교체**다

Q7-D~Q7-S 가 물어온 건 처음부터 하나였다 — **「리듬 너머로 P 창 형태가 정보를 주는가」**.
`f1`·`f2_k`·`f3`~`f6` 은 연구 대상이 아니라 **통제해야 할 교란원**이었다.

문제는 **형태 측정기**였다.

```
지금까지  R 기준 고정창 + 두 템플릿 거리   ← R24 가 「심박수 대리변수」라고 못 박은 도구
Q7-T      P delineation + QRST 소거 잔차   ← 같은 질문, 제대로 된 좌표계
```

### 왜 고정창이 형태를 못 재는가 — Q7-R 실측으로

```
P 피크 위치   R−158ms · IQR R−183 ~ −139ms   ← ±22ms(±8샘플) 흔들린다
창 폭         22샘플(61ms)                    ← P 가 창 폭만큼 움직인다
```

게다가 Q7-R 의 `p_peak()` 는 **`p_mid_22` 안에서** 최대 편차를 찾았다 — R 로 정의한 창
안에서 P 를 찾고 그걸 P 라 부르는 **순환**이다. 독립적인 delineation 이 없었다.

그래서 두 템플릿 거리의 상당 부분은 **모양 차이가 아니라 위치 차이**이고, 조기성 통제가
그 위치를 지우면서 **위치로 부호화된 형태 정보까지 같이 지웠다.**

## ★★ QRST 소거 — 숨은 P 를 잔차로 꺼낸다

「P 가 QRS·T 와 겹치면, QRS·T 의 변화(잡음처럼 보이지만)를 보면 된다」는 발상은
**확립된 방법**이다 — Stridh & Sörnmo, *Spatiotemporal QRST cancellation techniques for
analysis of atrial fibrillation*, IEEE TBME 48:105–111 (2001).

```
심실 활동(QRS+T)을 템플릿으로 적합 → 차감 → **잔차 = 주로 심방 활동**
시공간 판본(인접 유도 활용 · 전기축 변동 보정)이 단순 평균차감(ABS)보다 오차 42% 감소
```

이 노트북은 둘 다 구현해 **나란히 비교**한다.

- **`abs`** — 평균 비트 차감(median 템플릿)
- **`st`** — 시공간-lite: 비트마다 **두 유도 템플릿의 선형결합 + 시프트**를 최소제곱 적합
  (Stridh–Sörnmo 의 구조를 2유도로 축소한 것. **완전한 판본이 아니라고 명시한다**)

★ **적합은 심실 구간(QRS+T)에서만** 하고 **차감은 전 구간**에 한다 — P 구간이 적합에
끼면 P 를 같이 지운다.

## ★★ 그리고 이번엔 **외부 정답**이 있다

이 퀘스트의 도구는 전부 **자기검증**이었다(마스크·잔차화·층화·`dr`·양성 대조).
정답이 없으니 「이 도구가 맞나」를 도구 자신으로 물어야 했고 매번 미결이 쌓였다.

**[BUT PDB](https://physionet.org/content/but-pdb/1.0.0/)** 가 그걸 바꾼다.

```
50개 · 2분 · 2유도 · WFDB
출처: MIT-BIH Arrhythmia + **MIT-BIH Supraventricular Arrhythmia(SVDB)** + Long Term AF
전문가 **2명**이 전 레코드 P 피크 수동 주석 (불일치는 합의까지 재확인)
P파 **5,437**개 · QRS **7,638**개 중 **2,201개(28.8%)는 P 파가 없다**
```

**「P 없음」의 정답까지 있다** — Q7-R 까지의 표현으로는 아예 나타낼 수 없던 상태다.

그리고 **벤치마크**도 있다 — Saclova et al., *Reliable P wave detection in pathological
ECG signals*, Sci Rep 12:6589 (2022) 가 BUT PDB 에서 **Se 93.07% / PP 88.60%**.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **T1 ★★** | P 검출기 vs **전문가 주석** (허용 ±50ms) | **Se ≥ 0.70 AND PPV ≥ 0.70** · 벤치마크 93.07/88.60 병기 |
| **T2 ★★** | **QRST 소거가 P 가시성을 올리는가** — 주석 P 위치의 잔차 대비가 **소거 전보다** 큰가 | 개선분 CI 하한 > 0 · 레코드 클러스터 부트스트랩 |
| **T3 ★** | **「P 부재」 판별** — P 있는 QRS vs 없는 QRS | AUROC CI 하한 > **0.65** |
| **T4** | (관문 아님) SVDB 전이 — 검출률·부재율이 임상적으로 말이 되는가 · S vs N | 보고만 |

### 판정표 (R29 ②)

- **T1 ✅ · T2 ✅** → 자가 섰다. **Q7-S′**(같은 설계, `feats_for()` 만 P 정렬 특징으로)로 간다
- **T1 ❌ · 벤치마크와 격차 큼** → **자체 검출기를 버리고 공개 방법**(phasor transform ·
  CEEMDAN)을 쓴다. 이것도 결정 가능한 결과다
- **T2 ❌** → 소거가 P 를 못 꺼낸다. 「QRS·T 에 묻힌 P」 경로를 닫는다
- **T3 ✅** → **「P 부재」가 특징이 된다** — SVEB 의 non-conducted/은닉 P 를 처음으로 표현 가능
- **어느 것이든 ⛔ 측정 불가** → 어떤 결론 분기도 타지 않는다

⚠️ **이 런은 SVEB 질문에 답하지 않는다.** 「P 가 SVEB 판별에 도움이 되나」가 아니라
**「P 를 볼 수 있기는 한가」**만 답한다. 의도적으로 한 걸음 물러서는 런이다 —
그래야 이후 모든 런의 「미결」이 **측정 한계인지 효과 없음인지** 갈린다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    """최소 검출 효과 = CI 반폭(R33 ①)."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def need_n(n, lo, hi, mean, margin):
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    return None if slack <= 0 else float(n) * (((hi - lo) / 2.0) / slack) ** 2

def boot_mean(v, seed, nb=3000, q=2.5):
    """레코드 단위 부트스트랩 평균 + CI."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def match_1d(det, ref, tol):
    """검출 위치와 정답 위치를 **탐욕적 1:1** 로 맞춘다(중복 매칭 금지).
    반환 (매칭수, |오차| 배열). 1:1 을 안 걸면 하나의 검출이 여러 정답을 먹어 Se 가 부푼다."""
    det = np.sort(np.asarray(det, float)); ref = np.sort(np.asarray(ref, float))
    used = np.zeros(len(det), bool); errs = []
    for r in ref:
        cand = np.where((~used) & (np.abs(det - r) <= tol))[0]
        if len(cand):
            j = cand[np.argmin(np.abs(det[cand] - r))]
            used[j] = True; errs.append(float(det[j] - r))
    return len(errs), np.asarray(errs, float)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1

# ── 공통 좌표계 (SVDB 파이프라인과 동일하게 맞춘다)
FS, RPRE, BEAT_LEN = 360, 100, 300      # R = index 100 · 비트 길이 300샘플

# ── ★★ QRST 소거
FIT_LO, FIT_HI = 85, 250                # ★ **심실 구간에서만 적합**(P 구간이 끼면 P 를 지운다)
SHIFTS = tuple(range(-4, 5))            # 시공간-lite 의 비트별 시프트 탐색 범위(±11ms)
CANCEL = ("none", "abs", "st")          # 소거 없음 · 평균차감 · 시공간-lite

# ── P 탐색 구간 (R 기준 · Q7-Q/R 창과 같은 좌표)
P_LO, P_HI = 0, 85                      # R−278 ~ −42ms
# ★ **P_HI ≤ FIT_LO** 이어야 한다 — 겹치면 P 가 적합에 끼어 소거가 P 를 지운다.
assert P_HI <= FIT_LO, "P 탐색 구간이 적합 구간과 겹친다 — 소거가 P 를 지운다"
P_SMOOTH = 5                            # 잔차 평활 탭수
REF_LO, REF_HI = 250, 300               # 잡음 기준 구간(TP 분절 후반)

# ── ★★ 관문 사전등록
TOL_MS   = 50                           # P 위치 허용 오차(문헌 관행)
TOL      = int(round(TOL_MS * FS / 1000))
SE_MIN, PPV_MIN = 0.70, 0.70            # T1 — 자체 검출기 기준(공개 방법보다 낮게 잡는다)
BENCH_SE, BENCH_PP = 0.9307, 0.8860     # Saclova et al. 2022, BUT PDB
T3_AUROC_MIN = 0.65                     # T3 — P 부재 판별
BUT_DIR = "but-pdb/1.0.0"

RULE_CHECK = {
    "R16 fallback 없음":       "BUT PDB 다운로드 실패 시 **중단** — 합성으로 대체하지 않는다",
    "R17 최소 n":              "레코드별 P 주석 수 미달이면 그 레코드를 판정에서 뺀다",
    "R22 누수 없음":           "템플릿은 **개체 안**에서만 · 정답 주석은 **평가에만** 쓴다",
    "R27 ③ 좌표 정합":         "BUT PDB 를 **360Hz 로 리샘플**해 SVDB 와 같은 좌표계로",
    "R29 ② 측정 불가 분기 금지": "⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R30 ① 필요표본":          "미결이면 필요 레코드 수를 계산해 출력",
    "R33 ① MDE":               "관문마다 MDE(CI 반폭)를 내고 점추정과 비교",
    "R34 ① 판정 규칙 명시":    "★ Se/PPV 매칭은 **탐욕적 1:1** — 규칙을 코드와 출력에 박는다",
    "R34 ④ 문턱 근거":         "★ TOL=±50ms 는 문헌 관행 · SE_MIN 은 공개 방법 대비 보수적",
    "R34 ⑤ 종결 조건":         "T1 이 벤치마크와 크게 벌어지면 **자체 검출기를 버린다**",
}

CONFIG = dict(
    exp="quest46_q7t_p_anchored", quest="ailab-2026-0046", step="but-pdb-validate",
    parent_exp=["quest46_q7r_calibrate", "ailab-2026-0067"],
    purpose=("Q7-D~S 는 계속 「리듬 너머로 P 형태가 정보를 주는가」를 물었는데, 형태 "
             "측정기가 **R 기준 고정창 + 두 템플릿 거리**였다 — R24 가 심박수 대리변수라고 "
             "못 박은 그 도구다. Q7-R 실측에서 P 피크가 R−158ms·IQR ±22ms 로 창 폭만큼 "
             "흔들리고, `p_peak()` 는 R 로 정의한 창 안에서 P 를 찾는 순환이었다. "
             "그래서 좌표계를 바꾼다 — **P delineation + QRST 소거 잔차**. 그리고 이 "
             "퀘스트에서 **처음으로 외부 정답**(BUT PDB 전문가 P 주석 · SVDB 파생 신호 "
             "포함)에 자를 맞춘다. 이 런은 SVEB 질문에 답하지 않는다 — 「P 를 볼 수 "
             "있기는 한가」만 답한다"),
    dataset="BUT PDB(50×2분×2유도 · 전문가 2명 P 주석) + SVDB 전수(전이 확인용)",
    fs=FS, rpre=RPRE, beat_len=BEAT_LEN, fit_window=[FIT_LO, FIT_HI],
    p_window=[P_LO, P_HI], shifts=list(SHIFTS), cancel=list(CANCEL),
    tol_ms=TOL_MS, se_min=SE_MIN, ppv_min=PPV_MIN,
    benchmark=dict(paper="Saclova et al., Sci Rep 12:6589 (2022)",
                   se=BENCH_SE, pp=BENCH_PP, db="BUT PDB"),
    method_ref="Stridh & Sornmo, IEEE TBME 48:105-111 (2001) — spatiotemporal QRST cancellation",
    rule_check=RULE_CHECK,
    predictions={
        "T1": f"★★ P 검출기 vs **전문가 주석**(허용 ±{TOL_MS}ms · 탐욕적 1:1 매칭). "
              f"**Se ≥ {SE_MIN} AND PPV ≥ {PPV_MIN}**. ★ 검출기는 **기권할 수 있다** "
              "(LORO Youden 문턱) — 기권이 없으면 PPV 상한이 P 유병률로 고정돼 "
              "유병률을 재게 된다. 벤치마크 "
              f"Se {BENCH_SE:.4f}/PP {BENCH_PP:.4f} 를 나란히 낸다 — 크게 벌어지면 "
              "**자체 검출기를 버리고 공개 방법으로 간다**(그것도 결정 가능한 결과다)",
        "T2": "★★ **QRST 소거가 P 가시성을 올리는가** — 주석 P 위치의 **국소 두드러짐**"
              "(P 창 안 백분위 순위 · 우연 0.5)이 **소거 전보다** 큰가. 개선분 CI 하한 > 0. "
              "★ 절대 대비를 안 쓴다 — 소거 전엔 앞 T 꼬리가 창을 덮어 배경을 재게 된다",
        "T3": f"★ **「P 부재」 판별** — P 있는 QRS vs 없는 QRS 를 잔차 최대 대비로. "
              f"AUROC CI 하한 > {T3_AUROC_MIN}. BUT PDB 는 QRS 7,638 중 2,201(28.8%)이 "
              "P 없음 — Q7-R 까지의 표현으로는 나타낼 수 없던 상태다",
        "T4": "(관문 아님) SVDB 전이 — 같은 검출기·소거를 SVDB 에 걸어 검출률/부재율이 "
              "임상적으로 말이 되는지, **S vs N 에서 다른지** 보고만 한다"},
    caveat=("★ **`st` 는 Stridh–Sornmo 의 완전한 판본이 아니다** — 2유도로 축소한 "
            "시공간-lite(두 유도 템플릿의 선형결합 + 비트별 시프트)다. 그렇게만 부른다. "
            "★ **적합은 심실 구간에서만** 한다 — P 구간이 적합에 끼면 P 를 같이 지운다. "
            "★ 그래도 **차감은 전 구간**이라 템플릿의 P 성분(중앙값 P)은 같이 빠진다 — "
            "PR 이 비트마다 흔들려 중앙값 P 가 뭉개지는 만큼만 P 가 살아남는다. 이게 "
            "T2 가 실제로 재는 것이다(감쇠분 vs 배경 제거분의 순차익). "
            "★ **전문가 주석은 평가에만** 쓴다 — 검출기 튜닝에 쓰면 T1 이 무의미해진다. "
            "★ BUT PDB 는 50×2분이라 **SVDB 전체(78×30분)보다 훨씬 작다** — T1~T3 는 "
            "「자가 맞나」이지 「SVDB 에서 얼마나 잡히나」가 아니다. 그건 T4 가 본다. "
            "★ **이 런은 SVEB 질문에 답하지 않는다.** 학습 0회 · 예상 15~25분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7t_p_anchored", CONFIG, project=PROJECT)
run.log("설정 ✅ **외부 정답(BUT PDB)에 자를 맞추는 런** — SVEB 질문엔 답하지 않는다")
run.log(f"  좌표계 {FS}Hz · R=idx {RPRE} · 적합 구간 [{FIT_LO},{FIT_HI}] · P 탐색 [{P_LO},{P_HI}]")
run.log(f"  허용 오차 ±{TOL_MS}ms(={TOL}샘플) · 매칭 **탐욕적 1:1**")
run.log(f"  벤치마크 — Saclova 2022 BUT PDB: Se {BENCH_SE:.4f} / PP {BENCH_PP:.4f}")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<24} {v_}")

In [ ]:
# CELL 2 — 【T-0a】 BUT PDB 적재 · 리샘플 · 비트 절단 (fallback 없음 R16)
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb
from scipy.signal import resample_poly
from math import gcd

run.log("\n" + "=" * 100)
run.log("【T-0a】 BUT PDB — 전문가 P 주석 (외부 정답)")
run.log("=" * 100)
BUT_RECS = [str(r) for r in wfdb.get_record_list("but-pdb")]   # ⛔ 실패 시 중단(R16)
if len(BUT_RECS) < 10:
    raise AssetError(f"BUT PDB 레코드 목록이 {len(BUT_RECS)}개 — 다운로드 실패")
run.log(f"  레코드 {len(BUT_RECS)}개")

def find_ext(rid, cands):
    """주석 확장자 이름을 **탐색**한다(합성 대체가 아니라 이름 확인일 뿐 — R16 유지).
    후보를 다 못 찾으면 **중단**한다."""
    for e in cands:
        try:
            wfdb.rdann(rid, e, pn_dir=BUT_DIR); return e
        except Exception:
            continue
    raise AssetError(f"{rid}: 주석 확장자를 못 찾았다 — 후보 {cands}")

EXT_Q = find_ext(BUT_RECS[0], ["qrs", "atr", "ari"])
EXT_P = find_ext(BUT_RECS[0], ["pwave", "pwv", "p"])
run.log(f"  주석 확장자 — QRS `{EXT_Q}` · P `{EXT_P}`")

def cut_beats(sig, rpos, fs_in):
    """`sig`(n×2)를 360Hz 로 리샘플하고 R 위치마다 300샘플 비트를 자른다.
    반환 (beats(n×2×300), 새 R 위치, 리샘플 배율)."""
    sig = np.nan_to_num(np.asarray(sig, float), nan=0.0, posinf=0.0, neginf=0.0)
    g = gcd(int(FS), int(fs_in))
    x = resample_poly(sig, int(FS) // g, int(fs_in) // g, axis=0) if fs_in != FS else sig
    scale = FS / float(fs_in)
    rp = np.round(np.asarray(rpos, float) * scale).astype(int)
    keep = (rp >= RPRE) & (rp < len(x) - (BEAT_LEN - RPRE))
    rp = rp[keep]
    B = np.stack([x[p - RPRE:p - RPRE + BEAT_LEN, :2].T for p in rp]) if len(rp) else \
        np.zeros((0, 2, BEAT_LEN))
    return B.astype("float64"), rp, scale

BUT = {}
T0 = time.time()
for rid in BUT_RECS:
    rec = wfdb.rdrecord(rid, pn_dir=BUT_DIR)
    aq = wfdb.rdann(rid, EXT_Q, pn_dir=BUT_DIR)
    ap = wfdb.rdann(rid, EXT_P, pn_dir=BUT_DIR)
    sig = np.asarray(rec.p_signal, float)
    if sig.ndim != 2 or sig.shape[1] < 2:
        raise AssetError(f"{rid}: 2유도가 아니다 {sig.shape}")
    B, rp, sc = cut_beats(sig, aq.sample, rec.fs)
    if len(rp) < 5:
        continue
    p_res = np.round(np.asarray(ap.sample, float) * sc).astype(int)
    # 비트마다 P 탐색 구간에 든 **정답 P** 를 모은다(비트 상대 index)
    p_true, has_p = [], np.zeros(len(rp), bool)
    for i, p0 in enumerate(rp):
        lo, hi = p0 - RPRE + P_LO, p0 - RPRE + P_HI
        hit = p_res[(p_res >= lo) & (p_res < hi)]
        rel = (hit - (p0 - RPRE)).astype(int)
        p_true.append(rel)
        has_p[i] = len(rel) > 0
    BUT[rid] = dict(B=B, r=rp, p_true=p_true, has_p=has_p, fs_in=rec.fs,
                    n_p=int(len(p_res)))
run.log(f"  적재 {len(BUT)}개 · {time.time()-T0:.0f}초")
_fs = sorted({v["fs_in"] for v in BUT.values()})
_nb = sum(len(v["r"]) for v in BUT.values())
_np_ = sum(v["n_p"] for v in BUT.values())
_hp = sum(int(v["has_p"].sum()) for v in BUT.values())
run.log(f"  원 샘플레이트 {_fs} → **{FS}Hz 로 리샘플**(R27 ③ · SVDB 좌표계와 정합)")
run.log(f"  QRS **{_nb:,}** · 주석 P **{_np_:,}** · P 탐색창 안에 정답이 있는 비트 "
        f"**{_hp:,}**({_hp/max(_nb,1):.3f})")
run.log(f"  → **P 없는 QRS {_nb-_hp:,}개({1-_hp/max(_nb,1):.3f})** — T3 의 정답")
run.log("  (문헌값: QRS 7,638 중 2,201(28.8%)이 P 없음 · 창 정의 차이로 값이 다를 수 있다)")
CONFIG["but"] = dict(n_rec=len(BUT), n_qrs=int(_nb), n_p=int(_np_),
                     frac_with_p=float(_hp / max(_nb, 1)), fs_in=[int(f) for f in _fs])
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【T-A】 ★★ QRST 소거 (ABS · 시공간-lite)
# Stridh & Sornmo, IEEE TBME 48:105-111 (2001).
# ⚠️ `st` 는 **2유도로 축소한 시공간-lite** 다 — 완전한 판본이 아니다. 그렇게만 부른다.
def qrst_cancel(B, mode):
    """`B`(n×2×300) 에서 심실 활동을 빼고 잔차를 돌려준다.

    ★ **적합은 [FIT_LO, FIT_HI] 심실 구간에서만** 한다 — P 구간이 적합에 끼면 P 를
    같이 지운다. 차감은 전 구간에 한다.
    `abs` : median 템플릿을 그대로 차감(평균 비트 차감)
    `st`  : 비트마다 **두 유도 템플릿의 선형결합 + 시프트**를 최소제곱 적합
            (전기축 변동·진폭 변동을 흡수 → Stridh–Sornmo 의 구조를 축소한 것)"""
    if mode == "none":
        return B.copy()
    n, L = len(B), B.shape[2]
    T = np.median(B, axis=0)                       # (2, L) 템플릿
    if mode == "abs":
        return B - T[None]
    if mode != "st":
        raise AssetError(f"소거 방식 {mode} 를 모른다")
    fit = slice(FIT_LO, FIT_HI)
    best = np.full(n, np.inf)
    out = B - T[None]
    for d in SHIFTS:
        Ts = np.stack([np.roll(T[j], d) for j in range(2)])          # (2, L)
        X = np.c_[Ts[0, fit], Ts[1, fit], np.ones(FIT_HI - FIT_LO)]  # (m, 3)
        Pinv = np.linalg.pinv(X)                                     # (3, m)
        Xf = np.c_[Ts[0], Ts[1], np.ones(L)]                         # (L, 3)
        for l in range(2):
            C = Pinv @ B[:, l, fit].T                                # (3, n)
            R = B[:, l, :] - (Xf @ C).T                              # (n, L)
            if l == 0:
                res0 = R
            else:
                res1 = R
        err = (res0[:, fit] ** 2).sum(1) + (res1[:, fit] ** 2).sum(1)
        imp = err < best
        if imp.any():
            best[imp] = err[imp]
            out[imp, 0, :] = res0[imp]; out[imp, 1, :] = res1[imp]
    return out

def smooth(x, k):
    if k <= 1: return x
    ker = np.ones(k) / k
    return np.apply_along_axis(lambda v: np.convolve(v, ker, mode="same"), -1, x)

def p_detect(RES):
    """잔차에서 P 를 찾는다 — **정답 주석을 쓰지 않는다**(R22).
    반환 (P 위치 index, 대비 = |진폭|/잡음 MAD)."""
    E = np.abs(smooth(RES, P_SMOOTH)).sum(axis=1)          # 두 유도 합성 (n, L)
    seg = E[:, P_LO:P_HI]
    pk = P_LO + np.argmax(seg, axis=1)
    amp = seg.max(axis=1)
    bg = E[:, REF_LO:REF_HI]                               # 잡음 기준(정답과 무관)
    mad = np.median(np.abs(bg - np.median(bg, axis=1, keepdims=True)), axis=1) + 1e-9
    return pk, amp / mad

run.log("\n" + "=" * 100)
run.log("【T-A】 QRST 소거 — " + " · ".join(CANCEL))
run.log("=" * 100)
run.log(f"  적합 구간 idx [{FIT_LO}, {FIT_HI}) = R{(FIT_LO-RPRE)/FS*1000:+.0f}~"
        f"{(FIT_HI-RPRE)/FS*1000:+.0f}ms (심실) · 차감은 전 구간")
T1 = time.time()
DET = {c: {} for c in CANCEL}
for rid, d in BUT.items():
    for c in CANCEL:
        RES = qrst_cancel(d["B"], c)
        pk, con = p_detect(RES)
        DET[c][rid] = dict(pk=pk, con=con, res=RES)
run.log(f"  ({time.time()-T1:.0f}초) 소거별 잔차 에너지(심실 구간 RMS · 작을수록 잘 지움)")
for c in CANCEL:
    v = [float(np.sqrt((DET[c][rid]["res"][:, :, FIT_LO:FIT_HI] ** 2).mean()))
         for rid in BUT]
    run.log(f"    {c:<5} {np.mean(v):.5f}")
run.log("  ▸ `st` 가 `abs` 보다 작아야 시공간 보정이 작동한 것이다(문헌: ABS 대비 오차 42%↓)")
run.log("  ⚠️ 차감은 전 구간이라 **템플릿의 중앙값 P 성분도 같이 빠진다** — PR 이 흔들려")
run.log("     중앙값 P 가 뭉개지는 만큼만 P 가 남는다. T2 는 그 **순차익**(배경 제거분 −")
run.log("     P 감쇠분)을 재는 것이지 「P 가 그대로 보존된다」를 재는 게 아니다")
CONFIG["cancel_rms"] = {c: float(np.mean(
    [np.sqrt((DET[c][r]["res"][:, :, FIT_LO:FIT_HI] ** 2).mean()) for r in BUT]))
    for c in CANCEL}
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【T-B】 관문 T1 — 전문가 주석 대비 Se / PPV / 위치 오차
from sklearn.metrics import roc_curve
run.log("\n" + "=" * 100)
run.log(f"【T-B】 T1 — P 검출기 vs 전문가 주석 (허용 ±{TOL_MS}ms · **탐욕적 1:1 매칭**)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

# ★★ **검출기는 기권할 수 있어야 한다.** 모든 비트에서 후보를 하나씩 내면 PPV 의 상한이
#    「P 있는 비트 비율」로 **구조적으로 고정**된다(BUT PDB 는 ~0.71) — 그러면 PPV 문턱이
#    검출기 성능이 아니라 **유병률**을 재게 된다. 벤치마크(Saclova PP 0.8860)의 검출기는
#    기권한다. 그래서 여기서도 기권 문턱을 둔다.
# ★ 문턱은 **다른 레코드들에서만** 잡는다(LORO) — 자기 레코드 정답을 보면 T1 이 순환이다(R22).
def loro_thr(c, rid_out):
    """`rid_out` **을 뺀** 레코드들의 (대비, P유무)로 Youden 최적 기권 문턱을 잡는다."""
    cc, yy = [], []
    for rid, d in BUT.items():
        if rid == rid_out:
            continue
        cc.append(DET[c][rid]["con"]); yy.append(d["has_p"].astype(int))
    if not cc:
        return float("-inf")
    cc = np.concatenate(cc); yy = np.concatenate(yy)
    if yy.sum() < 5 or (1 - yy).sum() < 5:
        return float("-inf")                            # 층이 없으면 기권 안 함
    fpr, tpr, th = roc_curve(yy, cc)
    return float(th[int(np.argmax(tpr - fpr))])

T1TAB = {}
for c in CANCEL:
    se_v, pp_v, fire_v, err_all = [], [], [], []
    for rid, d in BUT.items():
        nref = sum(len(t) for t in d["p_true"])
        if nref < 5:
            continue                                    # R17 — 정답이 너무 적으면 뺀다
        thr = loro_thr(c, rid)                          # ★ 자기 레코드를 뺀 문턱
        fire = DET[c][rid]["con"] >= thr
        m_tot = 0; n_det = 0
        for i, ref in enumerate(d["p_true"]):
            det = [DET[c][rid]["pk"][i]] if fire[i] else []   # ★ 기권 가능
            n_det += len(det)
            if len(ref) and len(det):
                m, e = match_1d(det, ref, TOL)
                m_tot += m; err_all.extend(e.tolist())
        se_v.append(m_tot / max(nref, 1))
        pp_v.append(m_tot / max(n_det, 1))
        fire_v.append(float(fire.mean()))
    se_m, se_lo, se_hi, nse = boot_mean(se_v, SEED0 + 11)
    pp_m, pp_lo, pp_hi, _ = boot_mean(pp_v, SEED0 + 12)
    err = np.asarray(err_all, float) / FS * 1000.0
    T1TAB[c] = dict(se=se_m, se_lo=se_lo, se_hi=se_hi, pp=pp_m, pp_lo=pp_lo, pp_hi=pp_hi,
                    n=nse, fire=float(np.mean(fire_v)) if fire_v else float("nan"),
                    err_med=float(np.median(np.abs(err))) if len(err) else float("nan"),
                    err_iqr=[float(np.percentile(err, 25)), float(np.percentile(err, 75))]
                    if len(err) else [float("nan")] * 2)
    run.log(f"    {c:<5} Se **{se_m:.4f}** [{se_lo:.4f}, {se_hi:.4f}] · "
            f"PPV **{pp_m:.4f}** [{pp_lo:.4f}, {pp_hi:.4f}] · "
            f"발화율 {T1TAB[c]['fire']:.4f} · |오차| 중앙 {T1TAB[c]['err_med']:.1f}ms · n={nse}")
_fwp = CONFIG["but"]["frac_with_p"]
run.log(f"\n    ▸ 기권이 없으면 PPV 상한이 **{_fwp:.4f}**(=P 있는 비트 비율)로 고정된다 —")
run.log(f"      그러면 PPV 문턱 {PPV_MIN} 이 검출기가 아니라 **유병률**을 재게 된다. 그래서")
run.log("      **LORO Youden 기권 문턱**을 뒀다(자기 레코드 정답은 안 본다 — R22)")
BEST_C = max(T1TAB, key=lambda c: min(T1TAB[c]["se"], T1TAB[c]["pp"]))
b = T1TAB[BEST_C]
run.log(f"\n    ★ 최량 소거 = **`{BEST_C}`**")
run.log(f"    벤치마크 (Saclova 2022 · BUT PDB) — Se {BENCH_SE:.4f} / PP {BENCH_PP:.4f}")
run.log(f"    격차 — Se {b['se']-BENCH_SE:+.4f} · PPV {b['pp']-BENCH_PP:+.4f}")
if b["n"] < 3:
    g_("T1", "⛔ 측정 불가", "판정 가능한 레코드가 3 미만")
else:
    ok = (b["se_lo"] > SE_MIN) and (b["pp_lo"] > PPV_MIN)
    bad = (b["se_hi"] < SE_MIN) or (b["pp_hi"] < PPV_MIN)
    DIFF["T1"] = dict(cancel=BEST_C, **{k: b[k] for k in
                      ("se", "se_lo", "se_hi", "pp", "pp_lo", "pp_hi", "n", "err_med")})
    g_("T1", "✅ 지지" if ok else ("❌ 기각" if bad else "⚠️ 미결"),
       f"★★ `{BEST_C}` Se {b['se']:.4f} [{b['se_lo']:.4f}, {b['se_hi']:.4f}] · "
       f"PPV {b['pp']:.4f} [{b['pp_lo']:.4f}, {b['pp_hi']:.4f}] · 문턱 {SE_MIN}/{PPV_MIN}")
    run.log(f"       MDE — Se {mde(b['se_lo'], b['se_hi']):.4f} · "
            f"PPV {mde(b['pp_lo'], b['pp_hi']):.4f}")
CONFIG["T1"] = T1TAB; CONFIG["best_cancel"] = BEST_C
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【T-C】 관문 T2 (소거가 P 를 드러내는가) · T3 (P 부재 판별)
from sklearn.metrics import roc_auc_score
run.log("\n" + "=" * 100)
run.log("【T-C】 T2 — QRST 소거가 P 가시성을 올리는가 · T3 — 「P 부재」 판별")
run.log("=" * 100)

def p_prom(RES, rel_list):
    """**정답 P 위치의 국소 두드러짐** — 그 비트 P 창 안에서의 백분위 순위(0~1 · 우연 0.5).

    ★ **절대 대비(|진폭|/잡음 MAD)를 재면 안 된다.** 소거 전에는 **앞 비트 T 꼬리**가 P
      창을 덮고 있어서 「대비」가 되레 크게 나온다 — 그건 배경을 재는 것이지 P 를 재는
      게 아니다. 픽스처 ⑯ 이 이 함정을 실측으로 잡았다(합성에서 146.5 → 33.1, 즉
      소거가 P 를 드러냈는데도 절대 대비는 **떨어진다**).
      순위는 배경 크기에 **불변**이고 「P 가 창 안에서 두드러지는가」만 잰다."""
    E = np.abs(smooth(RES, P_SMOOTH)).sum(axis=1)
    W = E[:, P_LO:P_HI]
    out = []
    for i, rel in enumerate(rel_list):
        for p in rel:
            if P_LO <= p < P_HI:
                out.append(float((W[i] < E[i, p]).mean()))
    return out

run.log("  T2 — 정답 P 위치의 **국소 두드러짐**(P 창 백분위 순위 · 우연 0.5 · 레코드 평균)")
run.log("     ▸ 절대 대비가 아니라 **순위**를 쓴다 — 소거 전엔 앞 T 꼬리가 창을 덮어")
run.log("       절대 대비가 배경 크기를 재게 된다(픽스처 ⑯ 이 잡은 함정)")
GAIN = {}
for c in CANCEL:
    if c == "none":
        continue
    gains, lv0, lv1 = [], [], []
    for rid, d in BUT.items():
        b0 = p_prom(DET["none"][rid]["res"], d["p_true"])
        b1 = p_prom(DET[c][rid]["res"], d["p_true"])
        if len(b0) >= 5 and len(b1) >= 5:
            gains.append(float(np.mean(b1) - np.mean(b0)))
            lv0.append(float(np.mean(b0))); lv1.append(float(np.mean(b1)))
    m_, lo_, hi_, n_ = boot_mean(gains, SEED0 + 21)
    GAIN[c] = dict(mean=m_, lo=lo_, hi=hi_, n=n_,
                   lvl_none=float(np.mean(lv0)) if lv0 else float("nan"),
                   lvl=float(np.mean(lv1)) if lv1 else float("nan"))
    run.log(f"    {c:<5} 순위 {GAIN[c]['lvl_none']:.4f} → **{GAIN[c]['lvl']:.4f}** · "
            f"개선분 **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · n={n_} · MDE {mde(lo_, hi_):.4f}")
gb = max(GAIN, key=lambda c: GAIN[c]["mean"]) if GAIN else None
if gb is None or GAIN[gb]["n"] < 3:
    g_("T2", "⛔ 측정 불가", "개선분을 못 냈다")
else:
    d2 = GAIN[gb]
    DIFF["T2"] = dict(cancel=gb, **d2, mde=float(mde(d2["lo"], d2["hi"])))
    g_("T2", decide(d2["lo"], d2["hi"], 0.0, ">"),
       f"★★ 최량 `{gb}` 개선분 **{d2['mean']:+.4f}** [{d2['lo']:+.4f}, {d2['hi']:+.4f}] "
       f"· n={d2['n']}")
    run.log("       ▸ > 0 이면 **QRS·T 를 빼서 숨은 P 를 꺼낸다**는 발상이 실측으로 선다")
    run.log(f"       (소거 후 순위 {d2['lvl']:.4f} — 우연 0.5 · 1.0 이면 P 가 창 안 최대)")

run.log("\n  T3 — 「P 부재」 판별 (P 있는 QRS vs 없는 QRS · 잔차 최대 대비)")
T3 = {}
for c in CANCEL:
    au = []
    for rid, d in BUT.items():
        y = d["has_p"].astype(int)
        if y.sum() < 5 or (1 - y).sum() < 5:
            continue                                    # R17
        au.append(float(roc_auc_score(y, DET[c][rid]["con"])))
    m_, lo_, hi_, n_ = boot_mean(au, SEED0 + 31)
    T3[c] = dict(auroc=m_, lo=lo_, hi=hi_, n=n_)
    run.log(f"    {c:<5} AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_}")
tb = max(T3, key=lambda c: T3[c]["auroc"])
if T3[tb]["n"] < 3:
    g_("T3", "⛔ 측정 불가", "층이 있는 레코드가 3 미만")
else:
    d3 = T3[tb]
    DIFF["T3"] = dict(cancel=tb, **d3, mde=float(mde(d3["lo"], d3["hi"])))
    g_("T3", decide(d3["lo"], d3["hi"], T3_AUROC_MIN, ">"),
       f"★ 최량 `{tb}` AUROC **{d3['auroc']:.4f}** [{d3['lo']:.4f}, {d3['hi']:.4f}] "
       f"· 문턱 {T3_AUROC_MIN} · n={d3['n']}")
    run.log("       ▸ 지지면 **「P 부재」가 특징이 된다** — SVEB 의 은닉/비전도 P 를 처음으로 표현")
CONFIG["T2"] = GAIN; CONFIG["T3"] = T3
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【T-D】 T4 SVDB 전이 (관문 아님)
run.log("\n" + "=" * 100)
run.log("【T-D】 T4 — SVDB 전이 (관문 아님 · 보고만)")
run.log("=" * 100)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
BEAT = np.ascontiguousarray(np.asarray(d5["beat"])[keep]).astype("float64")
if BEAT.ndim != 3 or BEAT.shape[1:] != (2, BEAT_LEN):
    raise AssetError(f"SVDB 비트 모양이 (n,2,{BEAT_LEN}) 가 아니다 — {BEAT.shape}")
run.log(f"  SVDB 비트 {len(Y):,} · 레코드 {len(np.unique(REC))}")

# ★ 부재 문턱은 **BUT PDB 에서** 정한다 — SVDB 라벨을 보고 고르면 순환이다(R22)
con_all, y_all = [], []
for rid, d in BUT.items():
    con_all.extend(DET[BEST_C][rid]["con"].tolist()); y_all.extend(d["has_p"].astype(int).tolist())
con_all, y_all = np.asarray(con_all), np.asarray(y_all)
ABS_THR = float(np.percentile(con_all[y_all == 0], 75)) if (y_all == 0).any() else float("nan")
run.log(f"  ★ 부재 문턱 = BUT PDB 의 **P 없는 QRS 대비 75분위** = {ABS_THR:.3f} "
        "(SVDB 라벨을 안 보고 정한다 — 순환 방지)")
run.log("  ▸ 대비는 |진폭|/잡음 **비율**이라 신호 이득에 불변이다 — 그래도 두 DB 의 분포를")
run.log("    나란히 찍어 문턱이 **전이됐는지** 눈으로 확인한다(안 겹치면 부재율이 0/1 로 쏠린다)")

T2_ = time.time()
rows, SV_CON = [], []
for r in sorted(set(REC.tolist())):
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if tt.sum() < 20 or (~tt).sum() < 20:
        continue
    RES = qrst_cancel(BEAT[mm], BEST_C)
    pk, con = p_detect(RES)
    SV_CON.append(con)
    rows.append(dict(r=int(r),
                     abs_s=float((con[tt] < ABS_THR).mean()),
                     abs_n=float((con[~tt] < ABS_THR).mean()),
                     pos_s=float(np.median(pk[tt])), pos_n=float(np.median(pk[~tt]))))
run.log(f"  ({time.time()-T2_:.0f}초) 레코드 {len(rows)}개")
_qs = (5, 25, 50, 75, 95)
run.log(f"    대비 분위 {_qs} — BUT PDB "
        f"{np.round(np.percentile(con_all, _qs), 2).tolist()}")
run.log(f"    {'':<20}   SVDB     "
        f"{np.round(np.percentile(np.concatenate(SV_CON), _qs), 2).tolist()}")
_ov = float(((np.concatenate(SV_CON) >= np.percentile(con_all, 5)) &
             (np.concatenate(SV_CON) <= np.percentile(con_all, 95))).mean())
run.log(f"    → SVDB 대비의 **{_ov:.3f}** 가 BUT PDB 5~95분위 안에 든다 "
        f"({'전이 OK' if _ov > 0.5 else '⚠️ 분포가 어긋난다 — 부재율을 그대로 믿지 말 것'})")
for nm, key_s, key_n in (("P 부재율", "abs_s", "abs_n"), ("P 위치 중앙(idx)", "pos_s", "pos_n")):
    s_ = np.array([x[key_s] for x in rows]); n_ = np.array([x[key_n] for x in rows])
    m_, lo_, hi_, k_ = boot_mean(s_ - n_, SEED0 + 41)
    extra = ""
    if nm.startswith("P 위치"):
        extra = f"  (= R{(np.nanmean(s_)-RPRE)/FS*1000:+.0f}ms vs R{(np.nanmean(n_)-RPRE)/FS*1000:+.0f}ms)"
    run.log(f"    {nm:<16} S {np.nanmean(s_):.4f} · N {np.nanmean(n_):.4f} · "
            f"**차이 {m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}]{extra}")
run.log("  ▸ **관문이 아니다** — S/N 차이가 있어도 여기서 결론을 내지 않는다(Q7-S′ 가 묻는다)")
CONFIG["T4"] = dict(abs_thr=ABS_THR, n_rec=len(rows), overlap=_ov,
                    abs_s=float(np.nanmean([x["abs_s"] for x in rows])),
                    abs_n=float(np.nanmean([x["abs_n"] for x in rows])))
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【T-E】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))

# ① 소거 전/후 평균 잔차 파형 — P 구간이 드러나는가
rid0 = sorted(BUT)[0]
xs = (np.arange(BEAT_LEN) - RPRE) / FS * 1000.0
for c, col in zip(CANCEL, ("tab:gray", "tab:blue", "tab:red")):
    ax[0].plot(xs, np.abs(DET[c][rid0]["res"][:, 0, :]).mean(0), color=col, label=c, lw=1.0)
ax[0].axvspan((P_LO - RPRE) / FS * 1000, (P_HI - RPRE) / FS * 1000,
              color="tab:green", alpha=.12)
ax[0].axvline(0, color="k", lw=.8)
ax[0].set_xlabel("ms relative to R   (shaded = P search)")
ax[0].set_ylabel("mean |residual|, lead 0"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② T1 — Se/PPV vs 벤치마크
cs = list(CANCEL); w = 0.35
ax[1].bar(np.arange(len(cs)) - w / 2, [T1TAB[c]["se"] for c in cs], w, label="Se")
ax[1].bar(np.arange(len(cs)) + w / 2, [T1TAB[c]["pp"] for c in cs], w, label="PPV")
ax[1].axhline(BENCH_SE, ls="--", lw=1.0, color="tab:red")
ax[1].axhline(SE_MIN, ls=":", lw=1.0, color="k")
ax[1].text(len(cs) - 0.5, BENCH_SE, " Saclova 2022", fontsize=7, color="tab:red", va="bottom")
ax[1].set_xticks(range(len(cs))); ax[1].set_xticklabels(cs, fontsize=8)
ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("vs expert annotations")
ax[1].set_xlabel("QRST cancellation"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="y")

# ③ T2 개선분 · T3 AUROC
gs = [c for c in CANCEL if c in GAIN]
ax[2].errorbar([GAIN[c]["mean"] for c in gs], np.arange(len(gs)),
               xerr=[[GAIN[c]["mean"] - GAIN[c]["lo"] for c in gs],
                     [GAIN[c]["hi"] - GAIN[c]["mean"] for c in gs]],
               fmt="o", color="tab:blue", capsize=4)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_yticks(range(len(gs))); ax[2].set_yticklabels([f"T2 {c}" for c in gs], fontsize=8)
ax[2].set_xlabel("P prominence gain (percentile rank in P window)"); ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7t_p_anchored", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")   # R29 ②
for g in ("T1", "T2", "T3"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if any(un_(g) for g in ("T1", "T2", "T3")):
    run.log("  ⛔ 측정 불가가 있다 — 어떤 결론 분기도 타지 않는다(R29 ②)")
elif ok_("T1") and ok_("T2"):
    run.log("  ★★ **자가 섰다.** P delineation + QRST 소거가 외부 정답을 재현한다.")
    run.log("     → **Q7-S′** 로 간다 — Q7-S 설계에서 `feats_for()` 만 **P 정렬 특징**으로")
    run.log("     바꿔 재실행한다(개인 내 vs 교차환자를 해석 가능한 상태로)")
elif no_("T1"):
    b_ = DIFF.get("T1", {})
    run.log(f"  ⛔ **자체 검출기가 문턱에 못 미친다** (Se {b_.get('se', float('nan')):.4f} · "
            f"PPV {b_.get('pp', float('nan')):.4f} vs 벤치마크 {BENCH_SE:.4f}/{BENCH_PP:.4f}).")
    run.log("     → **자체 검출기를 버리고 공개 방법**(phasor transform · CEEMDAN)으로 간다.")
    run.log("     이것도 결정 가능한 결과다 — 더 튜닝하지 않는다")
elif not ok_("T2"):
    run.log("  ⛔ **소거가 P 를 못 꺼낸다** — 「QRS·T 에 묻힌 P」 경로를 닫는다.")
    run.log("     P 가 보이는 비트만으로 갈 것인지 다시 정한다")
else:
    run.log("  ⚠️ 미결 — MDE 와 비교해 「효과 없음」인지 「측정 한계」인지 먼저 가른다(R33 ①)")
    for g in ("T1", "T2", "T3"):
        d_ = DIFF.get(g)
        if d_ and "mde" in d_:
            nn = need_n(d_.get("n", 0), d_["lo"], d_["hi"], d_["mean"], 0.05) \
                 if "mean" in d_ else None
            if nn is not None and np.isfinite(nn):
                run.log(f"     {g} 필요 레코드 ≈ {nn:.0f} (현재 {d_.get('n')})")
if ok_("T3"):
    run.log("  ★ **「P 부재」가 특징이 된다** — Q7-R 까지의 표현으로는 나타낼 수 없던 상태다.")
    run.log(f"     SVDB 전이(T4): 부재율 S {CONFIG['T4']['abs_s']:.4f} vs "
            f"N {CONFIG['T4']['abs_n']:.4f} — **관문이 아니다**(Q7-S′ 가 묻는다)")
run.log("\n  ▸ 이 런은 **SVEB 질문에 답하지 않는다** — 「P 를 볼 수 있기는 한가」만 답했다")
run.log("  ▸ `st` 는 Stridh–Sornmo 의 **완전한 판본이 아니라 2유도 축소판**이다")

run.log("\n  사전등록 **종결 조건** (R34 ⑤) — 이 갈래를 언제 접는가")
run.log(f"    ① T1 이 벤치마크(Se {BENCH_SE:.4f})에서 0.15 이상 벌어지면 **자체 검출기를**")
run.log("       **버린다** — 더 튜닝하지 않고 공개 방법(phasor transform · CEEMDAN)으로 간다")
run.log("    ② T2 가 ❌ 면 「QRS·T 에 묻힌 P」 경로를 닫는다 — 소거 판본을 더 만들지 않는다")
run.log("    ③ T1·T2 가 둘 다 ⚠️ 미결이고 필요 레코드가 BUT PDB 50개를 넘으면, 같은 자료로")
run.log("       **더 돌지 않는다** — LUDB·QT DB 로 정답을 늘리거나 갈래를 접는다")
_gap = (BENCH_SE - DIFF.get("T1", {}).get("se", float("nan")))
run.log(f"    → 이번 격차 {_gap:+.4f} · 종결 ① {'발동' if _gap > 0.15 else '미발동'}")

run.finish({
    "exp_id": "quest46_q7t_p_anchored",
    "metric": "butpdb_p_detect_se",
    "value": float(DIFF.get("T1", {}).get("se", float("nan"))),
    "passed": bool(ok_("T1") and ok_("T2")),
    "summary": ("BUT PDB 전문가 P 주석을 외부 정답으로 삼아 P delineation + QRST 소거 "
                "측정기를 검증했다. SVEB 질문에는 답하지 않는다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "T1": CONFIG.get("T1", {}), "T2": CONFIG.get("T2", {}), "T3": CONFIG.get("T3", {}),
    "T4": CONFIG.get("T4", {}), "but": CONFIG.get("but", {}),
    "cancel_rms": CONFIG.get("cancel_rms", {}), "best_cancel": BEST_C,
    "benchmark": CONFIG.get("benchmark", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step but-pdb-validate`")